In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
model= ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite-preview')

In [18]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime)-> str: 
    '''function to read email content'''
    return runtime.state['email']

@tool
def send_email(body : str)-> str: 
    '''function to send the email with given body'''
    return body

In [19]:
from langchain.agents import AgentState, create_agent


class EmailState(AgentState):
    email: str


In [38]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
agent = create_agent(model = model, checkpointer = InMemorySaver(), tools = [read_email, send_email],
                     state_schema = EmailState,
                    system_prompt="You are an email assistant, use tools whenever necessary",
                    middleware = [
                        HumanInTheLoopMiddleware(interrupt_on = {
                            "read_email": False, #dont invoe HITL
                            "send_email":True #envoke HITL
                        },
                
                        description_prefix="Awating approval for tool call",
                                                      ),
                    ])

In [55]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "5"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [56]:
response

{'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='ff5b86dd-64a2-469f-9d40-95a4eb2407bc'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'69f9e05f-7728-49da-a561-9a3d0b9f03fe': 'EjQKMgEMOdbHxU2iDwWlsey0mBYJr/UAzDKohKjdnXuHfIiJ/WwcWerOnkwnTBPekPbg+tdF'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dc0e8-506b-7f33-9c64-e91727c7a848-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '69f9e05f-7728-49da-a561-9a3d0b9f03fe', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 10, 'total_tokens': 103, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="Hi Seán, I'm going to be la

In [57]:
response['__interrupt__']

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John,\n\nThanks for letting me know. That's no problem at all. Let me know what time works best for you to reschedule.\n\nBest,\nSeán"}, 'description': 'Awating approval for tool call\n\nTool: send_email\nArgs: {\'body\': "Hi John,\\n\\nThanks for letting me know. That\'s no problem at all. Let me know what time works best for you to reschedule.\\n\\nBest,\\nSeán"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='01de6314c07791a74af22ea0ba5421c8')]

In [58]:
response['__interrupt__'][0].value['action_requests'][0]['args']

{'body': "Hi John,\n\nThanks for letting me know. That's no problem at all. Let me know what time works best for you to reschedule.\n\nBest,\nSeán"}

In [32]:
from langgraph.types import Command
response = agent.invoke(Command(resume = {'decisions':[{'type': 'approve'}]}), config = config)
print(response)

{'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='71f58300-d92c-4049-9a4c-3c08344d3d8e'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'4267dd99-bbe0-4cdf-96fa-b38c77c09191': 'EjQKMgEMOdbH+ZSQrcqD4IPN7rPPe5+4F11GWRUHS9sBNGwPur1ZsQMLK7fGzjU78T/e6Lp1'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dc0cf-8d37-7c40-8497-2e97fce53033-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '4267dd99-bbe0-4cdf-96fa-b38c77c09191', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 10, 'total_tokens': 103, 'input_token_details': {'cache_read': 0}}), ToolMessage(content="Hi Seán, I'm going to be late f

In [48]:
from langgraph.types import Command
response = agent.invoke(Command(resume = {'decisions':[{'type': 'reject'}]}), config = config)
print(response)

{'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='9414aa02-9b63-40e4-b3b8-fe8e68947fe8'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'f8287242-c91a-4805-8de2-338c7e2716c1': 'EjQKMgEMOdbHKkwzhR5if7gpTOW9exh2TT3syoqQQ6T9OzSn4EoRvPWJ/WNMmQLWk44RBTQX'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dc0e1-8d2e-7362-b31f-6ddff2195a59-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'f8287242-c91a-4805-8de2-338c7e2716c1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 10, 'total_tokens': 103, 'input_token_details': {'cache_read': 0}}), ToolMessage(content="Hi Seán, I'm going to be late f

In [60]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

print(response)

{'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='ff5b86dd-64a2-469f-9d40-95a4eb2407bc'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'69f9e05f-7728-49da-a561-9a3d0b9f03fe': 'EjQKMgEMOdbHxU2iDwWlsey0mBYJr/UAzDKohKjdnXuHfIiJ/WwcWerOnkwnTBPekPbg+tdF'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dc0e8-506b-7f33-9c64-e91727c7a848-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '69f9e05f-7728-49da-a561-9a3d0b9f03fe', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 10, 'total_tokens': 103, 'input_token_details': {'cache_read': 0}}), ToolMessage(content="Hi Seán, I'm going to be late f

In [61]:
response['messages'][-1]

AIMessage(content=[{'type': 'text', 'text': 'The email has been sent.', 'extras': {'signature': 'EjQKMgEMOdbHG1GHRQ2VyoYfp2ULbBgT80B43Ut6uiujltAfqFPuNekWH18GDfQt3FiX3Nk1'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dc0e8-82da-7592-abdf-579df4e1d11f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 190, 'output_tokens': 6, 'total_tokens': 196, 'input_token_details': {'cache_read': 0}})